# Solution C Demo Inference

This notebook is the marker-facing inference path for the locked `Solution C` submission system.

Locked record: `experiments/runs/C_REMOTE_A40_019_transfer_seed3/result.json`

Official threshold: `0.50`


In [ ]:
# Run this once in a fresh environment.
%pip install -r requirements.txt sentencepiece


## Configure Paths

Set `SEED_MODEL_DIRS` to the three locked `Solution C` model directories.
If the final model artifacts are stored on the cloud, download them first and point these variables at the local `best_model` folders.


In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd()
GROUP_NUMBER = '52'
TOKENIZER_NAME = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
SEED_MODEL_DIRS = [
    REPO_ROOT / 'official_coursework' / 'local_artifacts' / 'C_REMOTE_A40_019_transfer_seed3' / 'seed_42' / 'best_model',
    REPO_ROOT / 'official_coursework' / 'local_artifacts' / 'C_REMOTE_A40_019_transfer_seed3' / 'seed_52' / 'best_model',
    REPO_ROOT / 'official_coursework' / 'local_artifacts' / 'C_REMOTE_A40_019_transfer_seed3' / 'seed_62' / 'best_model',
]
INPUT_CSV = REPO_ROOT / 'official_coursework' / 'released_test_data' / 'test_data' / 'ED' / 'test.csv'
OUTPUT_CSV = REPO_ROOT / 'outputs' / 'submission' / 'c_predictions_debug.csv'
SUBMISSION_FILE = REPO_ROOT / 'outputs' / 'submission' / f'Group_{GROUP_NUMBER}_C.csv'
THRESHOLD = 0.50
BATCH_SIZE = 16


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

if torch.backends.mps.is_available():
    device = 'mps'
elif torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'
print('Using device:', device)

for path in SEED_MODEL_DIRS:
    if not path.exists():
        raise FileNotFoundError(path)

df = pd.read_csv(INPUT_CSV)
texts_a = df['Claim'].astype(str).tolist()
texts_b = df['Evidence'].astype(str).tolist()

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=False)
encodings = tokenizer(texts_a, texts_b, truncation=True, max_length=256)

class PairDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        return {k: self.encodings[k][idx] for k in self.encodings}

def logits_to_pos_probs(logits: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits, dtype=np.float64)
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_vals = np.exp(shifted)
    probs = exp_vals / np.sum(exp_vals, axis=1, keepdims=True)
    return probs[:, 1]

dataset = PairDataset(encodings)
collator = DataCollatorWithPadding(tokenizer=tokenizer)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)

all_probs = []
for model_dir in SEED_MODEL_DIRS:
    print('Loading', model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))
    model.to(device)
    model.eval()
    seed_logits = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits.detach().cpu().numpy()
            seed_logits.append(logits)
    seed_logits = np.concatenate(seed_logits, axis=0)
    seed_probs = logits_to_pos_probs(seed_logits)
    all_probs.append(seed_probs)
    del model
    if device == 'mps':
        torch.mps.empty_cache()
    elif device == 'cuda':
        torch.cuda.empty_cache()

ensemble_probs = np.mean(np.vstack(all_probs), axis=0)
preds = (ensemble_probs >= THRESHOLD).astype(int)

out_df = df.copy()
out_df['prob_relevant'] = ensemble_probs
out_df['pred'] = preds
out_df.to_csv(OUTPUT_CSV, index=False)
out_df[['pred']].to_csv(SUBMISSION_FILE, index=False, header=False)
print({'rows': len(out_df), 'debug_csv': str(OUTPUT_CSV), 'submission_csv': str(SUBMISSION_FILE), 'positive_rate': float(preds.mean())})


In [ ]:
print('Detailed output exists:', OUTPUT_CSV.exists())
print('Submission file exists:', SUBMISSION_FILE.exists())
if SUBMISSION_FILE.exists():
    with open(SUBMISSION_FILE) as f:
        for _ in range(5):
            line = f.readline()
            if not line:
                break
            print(line.strip())
